# 🌐 WiseNet V1.5 — Pipeline Complet
### Optimisation Multi-Secteurs × Multi-Porteuses avec MILP 3GPP + CAMARA

Ce notebook orchestre le pipeline complet V1.5 :
1. **Topologie V1.5** : Sites avec 3 secteurs (0°, 120°, 240°) × 2 porteuses (F1=1.8 GHz, F2=3.5 GHz)
2. **Simulateur Spatial V1.5** : RSRP 3GPP, gain directionnel, path-loss dépendant de la fréquence
3. **Moteur MILP V1.5** : Optimisation des offsets → délestage Horizontal & Vertical
4. **Client CAMARA** : Intégration API GSMA Open Gateway (Network Insights + QoD)

> **Continuation directe de V1** — La V1 est préservée sur la branche `v1-stable` et le tag `v1.0-validated`.

## Phase 1 — Génération de la Topologie V1.5

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')

from src.topology.builder_v1_5 import TopologyBuilderV15

# Génération de la topologie sur un bloc de 50x50 mailles (zone test)
builder = TopologyBuilderV15(
    grid_cells_per_side=50,
    cell_size_meters=235.0,
    tx_power_dBm_F1=43.0,
    tx_power_dBm_F2=40.0
)

topology = builder.generate_topology(
    row_range=(20, 30),  # bloc de 10x10 = 100 mailles pour test
    col_range=(20, 30),
    density=0.3           # 30% des mailles ont un site
)

print(f'Sites générés : {len(topology)}')
print(f'Cellules radio (s,f) : {len(topology) * 3 * 2}  (3 secteurs × 2 porteuses par site)')

# Affichage d'un site exemple
first_site_id = next(iter(topology))
s = topology[first_site_id]
print(f'\nSite exemple : {first_site_id}')
for sec_name, sec in s['sectors'].items():
    print(f'  Secteur {sec_name} (azimuth {sec["azimuth_deg"]}°)')
    for cell_name, cell in sec['carriers'].items():
        print(f'    -> {cell["cell_id"]} | {cell["freq_ghz"]} GHz | Cap={cell["capacity_mo"]:.0f} Mo')

## Phase 2 — Simulation Spatiale RSRP 3GPP

In [ ]:
from src.spatial.simulator_v1_5 import SpatialTransferSimulatorV15

# Toutes les mailles du bloc 10x10
test_squares = [r * 50 + c + 1 for r in range(20, 30) for c in range(20, 30)]

sim = SpatialTransferSimulatorV15(
    grid_resolution=25,    # Points d'intégration par maille (25x25)
    cell_size_meters=235.0,
    delta_levels=[0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
)

fractions_data = sim.compute_transfer_fractions(test_squares, topology, grid_size=50)

print(f'Fractions précalculées pour {len(fractions_data)} mailles.')
print()

# Affichage d'un exemple
if fractions_data:
    sq_key = next(iter(fractions_data))
    sq_info = fractions_data[sq_key]
    print(f'Maille {sq_key} :')
    print(f'  Cellule maîtresse : {sq_info["master_cell"]}')
    for d, tdata in sq_info['offsets'].items():
        stays_pct = tdata['stays'] * 100
        targets = list(tdata['target_cells'].keys())
        print(f'  Offset {d} dB → reste={stays_pct:.1f}% | cibles={targets}')

## Phase 3 — Prédiction du Trafic Simulé

In [ ]:
import numpy as np

# Génération de trafic simulé : distribution normale autour de 12 000 Mo/maille
rng = np.random.default_rng(seed=42)
traffic_values = rng.normal(loc=12_000, scale=4_000, size=len(test_squares)).clip(min=1_000)

predicted_traffic = {
    str(sq): float(traffic_values[i])
    for i, sq in enumerate(test_squares)
}

total_traffic = sum(predicted_traffic.values())
print(f'Trafic total simulé    : {total_traffic:,.0f} Mo = {total_traffic/1024:.1f} Go')
print(f'Trafic moyen / maille  : {total_traffic/len(predicted_traffic):,.0f} Mo')

## Phase 4 — Capacités par Cellule Radio (s, f)

In [ ]:
cells_capacity = {}
for site_id, s_data in topology.items():
    for sec_id, sec_data in s_data['sectors'].items():
        for c_name, c_data in sec_data['carriers'].items():
            cells_capacity[c_data['cell_id']] = c_data['capacity_mo']

total_capacity = sum(cells_capacity.values())
print(f'Cellules radio (s,f)   : {len(cells_capacity)}')
print(f'Capacité réseau totale : {total_capacity:,.0f} Mo = {total_capacity/1024:.1f} Go')
print(f'Charge réseau          : {(total_traffic/total_capacity)*100:.1f}%  (> 100% = saturation)')

## Phase 5 — Optimisation MILP V1.5

In [ ]:
from src.optimization.milp_engine_v1_5 import MilpEngineV15

engine = MilpEngineV15()

result = engine.build_and_solve(
    predicted_traffic=predicted_traffic,
    fractions_data=fractions_data,
    cells_capacity=cells_capacity,
    delta_levels=[0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
)

print('═' * 55)
print('  📊 RÉSULTATS OPTIMISATION MILP V1.5')
print('═' * 55)
print(f'  Statique (sans opt.) : {result["static_unsatisfied_mo"]:>10,.0f} Mo')
print(f'  MILP (avec opt.)     : {result["optimized_unsatisfied_mo"]:>10,.0f} Mo')
print(f'  Réduction congestion : {result["gain_percentage"]:>9.1f} %')
print('═' * 55)

# Détail des cellules qui ont changé d'offset
switched = {c: d for c, d in result['decisions'].items() if d['offset_dB'] > 0.0}
print(f'\nCellules ayant basculé (offset > 0) : {len(switched)} / {len(result["decisions"])}')
for cell, dec in list(switched.items())[:5]:
    print(f'  {cell}: Δ={dec["offset_dB"]} dB | Congestion résiduelle={dec["residual_congestion_mo"]} Mo')

## Phase 6 — Intégration CAMARA (Network Insights + QoD)

In [ ]:
from src.camara.client import CamaraClient

# Initialisation en mode Sandbox Mock (pas besoin de credentials réels)
camara = CamaraClient(mock_mode=True)

# 1. Récupération des métriques réseau temps réel
insights = camara.get_network_insights(area_id='bloc_v1_5_test')
print('📡 Network Insights :')
for key, val in insights.get('metrics', {}).items():
    print(f'  {key}: {val}')

# 2. Simulation de session QoD pour un utilisateur en zone congestionnée
# On identifie les cellules avec congestion résiduelle la plus élevée
congested_cells = sorted(
    [(c, d['residual_congestion_mo']) for c, d in result['decisions'].items()],
    key=lambda x: x[1], reverse=True
)[:3]

print(f'\n⚠️  Top 3 cellules les plus congestionnées :')
for cell_name, cong_mo in congested_cells:
    print(f'  {cell_name}: {cong_mo:.0f} Mo résiduel')
    if cong_mo > 0:
        qod_resp = camara.request_qod_session(
            user_phone='+33601234567',
            qos_profile='QOS_EMERGENCY',
            duration_seconds=1800
        )
        print(f'    -> QoD activé : {qod_resp["sessionId"]} | Status: {qod_resp["qosStatus"]}')

## Phase 7 — Visualisation des Résultats

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Carte des offsets choisis par cellule
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Graphe 1 : Résiduel de congestion par cellule ---
cell_names_short = [c.split('_sec')[1] for c in result['decisions'].keys()]
cong_values = [d['residual_congestion_mo'] for d in result['decisions'].values()]

ax1 = axes[0]
colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in cong_values]
ax1.bar(range(len(cong_values)), cong_values, color=colors, edgecolor='white', linewidth=0.5)
ax1.set_title('Congestion résiduelle par cellule radio (s,f)', fontweight='bold', fontsize=11)
ax1.set_xlabel('Cellule radio (index)')
ax1.set_ylabel('Congestion résiduelle (Mo)')
ax1.axhline(0, color='gray', linestyle='--', linewidth=0.8)
green_patch = mpatches.Patch(color='#2ecc71', label='Satisfaite')
red_patch = mpatches.Patch(color='#e74c3c', label='Congestionnée')
ax1.legend(handles=[green_patch, red_patch])

# --- Graphe 2 : Distribution des offsets choisis ---
ax2 = axes[1]
offset_vals = [d['offset_dB'] for d in result['decisions'].values()]
unique_offsets, counts = np.unique(offset_vals, return_counts=True)
ax2.bar([f'{o:.1f}dB' for o in unique_offsets], counts,
        color='#3498db', edgecolor='white', linewidth=0.5)
ax2.set_title('Distribution des offsets de délestage choisis', fontweight='bold', fontsize=11)
ax2.set_xlabel('Niveau d\'offset (dB)')
ax2.set_ylabel('Nombre de cellules')

fig.suptitle(
    f'WiseNet V1.5 — Résultat MILP | Réduction congestion: {result["gain_percentage"]}%\n'
    f'Statique: {result["static_unsatisfied_mo"]:,.0f} Mo → MILP: {result["optimized_unsatisfied_mo"]:,.0f} Mo',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('../../research/reports/figures/v1_5_milp_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée dans research/reports/figures/v1_5_milp_results.png')

## ✅ Résumé du Pipeline V1.5

| Module | Fichier | Statut |
|--------|---------|--------|
| Topologie V1.5 | `src/topology/builder_v1_5.py` | ✅ Opérationnel |
| Simulateur Spatial | `src/spatial/simulator_v1_5.py` | ✅ Opérationnel |
| Moteur MILP | `src/optimization/milp_engine_v1_5.py` | ✅ Opérationnel |
| Client CAMARA | `src/camara/client.py` | ✅ Opérationnel (Mock + Prêt Prod) |

**Prochaines étapes :**
- Valider sur le bloc complet 1024 mailles (32×32)
- Intégrer les données trafic réelles du modèle LSTM/XGBoost V1
- Comparer V1 vs V1.5 sur les mêmes données de référence
- Obtenir des credentials CAMARA réels pour la démo GSMA